<a href="https://colab.research.google.com/github/naparnaanizham111-star/text-summarization-deep-learning/blob/main/tsn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets evaluate rouge_score sentencepiece accelerate matplotlib pandas


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
import datasets
import evaluate

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch version: 2.11.0+cu130
Transformers version: 5.18.0
CUDA available: True
GPU: Tesla T4


In [ ]:
from datasets import load_dataset
dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")
print(dataset)


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

3.0.0/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})


In [ ]:
print("Training samples:", len(dataset["train"]))
print("Validation samples:", len(dataset["validation"]))
print("Test samples:", len(dataset["test"]))
print("\nExample article:", dataset["train"][0]["article"][:500], "...")
print("\nReference summary:", dataset["train"][0]["highlights"])


Training samples: 287113
Validation samples: 13368
Test samples: 11490

Example article: LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as s ...

Reference summary: Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


In [ ]:
from transformers import AutoTokenizer
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Tokenizer loaded successfully!")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Tokenizer loaded successfully!


In [ ]:
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    inputs = ["summarize: " + article for article in examples["article"]]
    targets = examples["highlights"]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=max_target_length, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Preprocessing function ready.")


Preprocessing function ready.


In [ ]:
train_dataset = dataset["train"].select(range(10000))
validation_dataset = dataset["validation"].select(range(1000))
test_dataset = dataset["test"].select(range(1000))
print("Training samples:", len(train_dataset))
print("Validation samples:", len(validation_dataset))
print("Test samples:", len(test_dataset))

Training samples: 10000
Validation samples: 1000
Test samples: 1000


In [ ]:
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_validation = validation_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)
print("Tokenization completed!")


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenization completed!


In [ ]:
from transformers import T5ForConditionalGeneration
model = T5ForConditionalGeneration.from_pretrained("t5-small")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print("Using device:", device)


model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Using device: cuda


In [ ]:
tokenized_train = tokenized_train.remove_columns(["article", "highlights", "id"])
tokenized_validation = tokenized_validation.remove_columns(["article", "highlights", "id"])
tokenized_test = tokenized_test.remove_columns(["article", "highlights", "id"])
print("Unnecessary raw-text columns removed.")

Unnecessary raw-text columns removed.


In [ ]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
print("Data collator created successfully.")

Data collator created successfully.


In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-summarization",
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    report_to="none"
)
print("Training configuration created.")


Training configuration created.


In [ ]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.asarray(predictions)
    if predictions.ndim == 3:
        predictions = predictions.argmax(axis=-1)
    predictions = predictions.astype(np.int64)
    labels = np.where(labels == -100, tokenizer.pad_token_id, labels).astype(np.int64)
    predictions = np.clip(predictions, 0, tokenizer.vocab_size - 1)

    decoded_predictions = tokenizer.batch_decode(predictions.tolist(), skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels.tolist(), skip_special_tokens=True)

    scores = rouge.compute(
        predictions=decoded_predictions,
        references=decoded_labels,
        use_stemmer=True
    )
    return {
        "rouge1": scores["rouge1"],
        "rouge2": scores["rouge2"],
        "rougeL": scores["rougeL"]
    }

print("ROUGE evaluation function ready.")

ROUGE evaluation function ready.


In [ ]:
from transformers import Seq2SeqTrainer

# Keep the original training setup: 200 validation examples during training.
validation_for_training = tokenized_validation.select(range(200))

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=validation_for_training,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
print("Trainer created successfully.")


Trainer created successfully.


In [ ]:
train_result = trainer.train()
print("Training completed.")
print(train_result.metrics)

Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
500,1.178422,0.829026,0.041403,0.018510,0.035428
1000,1.175377,0.806774,0.197733,0.074832,0.160038
1500,1.144282,0.804117,0.234044,0.091246,0.190221
2000,1.119202,0.803930,0.237598,0.096317,0.192843
2500,1.090337,0.802615,0.236307,0.096175,0.193547


/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1954: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training completed.
{'train_runtime': 482.614, 'train_samples_per_second': 20.72, 'train_steps_per_second': 5.18, 'total_flos': 1353418014720000.0, 'train_loss': 1.304867837524414, 'epoch': 1.0}


In [ ]:
from collections import Counter

def token_overlap_metrics(pred_ids, label_ids):
    """Auxiliary token-level metrics for generated summaries."""
    pred_ids = np.asarray(pred_ids, dtype=np.int64)
    label_ids = np.asarray(label_ids, dtype=np.int64)
    label_ids = np.where(label_ids == -100, tokenizer.pad_token_id, label_ids)
    pred_ids = np.clip(pred_ids, 0, tokenizer.vocab_size - 1)

    total_overlap = 0
    total_pred_tokens = 0
    total_ref_tokens = 0
    correct_tokens = 0
    compared_tokens = 0
    squared_error = 0.0

    for pred, ref in zip(pred_ids, label_ids):
        pred_tokens = [int(x) for x in pred if int(x) != tokenizer.pad_token_id]
        ref_tokens = [int(x) for x in ref if int(x) != tokenizer.pad_token_id]

        overlap = sum((Counter(pred_tokens) & Counter(ref_tokens)).values())
        total_overlap += overlap
        total_pred_tokens += len(pred_tokens)
        total_ref_tokens += len(ref_tokens)

        n = min(len(pred_tokens), len(ref_tokens))
        if n > 0:
            pred_arr = np.asarray(pred_tokens[:n], dtype=np.float64)
            ref_arr = np.asarray(ref_tokens[:n], dtype=np.float64)
            squared_error += np.sum((pred_arr - ref_arr) ** 2)
            correct_tokens += np.sum(pred_arr == ref_arr)
            compared_tokens += n

    precision = total_overlap / total_pred_tokens if total_pred_tokens else 0.0
    recall = total_overlap / total_ref_tokens if total_ref_tokens else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall)

    return {
        "token_precision": float(precision),
        "token_recall": float(recall),
        "token_f1": float(f1),


    }

final_test_result = trainer.predict(
    tokenized_test,
    max_length=128,
    num_beams=4
)

predictions = np.asarray(final_test_result.predictions)
if predictions.ndim == 3:
    predictions = predictions.argmax(axis=-1)

labels = np.asarray(final_test_result.label_ids)

aux_metrics = token_overlap_metrics(predictions, labels)

evaluation_results = {
    key: float(value)
    for key, value in final_test_result.metrics.items()
    if isinstance(value, (int, float, np.integer, np.floating))
}
evaluation_results.update(aux_metrics)

print("FINAL TEST EVALUATION")
print("-" * 40)
for key, value in evaluation_results.items():
    print(f"{key}: {value:.6f}")


SyntaxError: expected 'else' after 'if' expression (2954550035.py, line 36)

In [ ]:
os.makedirs("./results/plots", exist_ok=True)

pd.DataFrame([evaluation_results]).to_csv(
    "./results/evaluation_results.csv",
    index=False
)

import json
with open("./results/evaluation_results.json", "w") as f:
    json.dump(evaluation_results, f, indent=4)

print("Evaluation results saved.")


In [ ]:
history = pd.DataFrame(trainer.state.log_history)
print("Available history columns:", history.columns.tolist())
history.to_csv("./results/training_history.csv", index=False)

# 1. Training and validation loss
plt.figure(figsize=(9, 5))
if "loss" in history.columns:
    h = history.dropna(subset=["loss"])
    plt.plot(h["step"], h["loss"], marker="o", label="Training Loss")
if "eval_loss" in history.columns:
    h = history.dropna(subset=["eval_loss"])
    plt.plot(h["step"], h["eval_loss"], marker="o", label="Validation Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("./results/plots/loss_curve.png", dpi=200)
plt.show()

# 2. ROUGE over evaluation steps
plt.figure(figsize=(9, 5))
for metric in ["rouge1", "rouge2", "rougeL"]:
    if metric in history.columns:
        h = history.dropna(subset=[metric])
        plt.plot(h["step"], h[metric], marker="o", label=metric.upper())
plt.xlabel("Training Steps")
plt.ylabel("ROUGE Score")
plt.title("ROUGE Scores During Training")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("./results/plots/rouge_curve.png", dpi=200)
plt.show()

# 3. Final ROUGE comparison
rouge_values = [
    evaluation_results.get("test_rouge1", evaluation_results.get("rouge1", 0)),
    evaluation_results.get("test_rouge2", evaluation_results.get("rouge2", 0)),
    evaluation_results.get("test_rougeL", evaluation_results.get("rougeL", 0))
]
plt.figure(figsize=(8, 5))
plt.bar(["ROUGE-1", "ROUGE-2", "ROUGE-L"], rouge_values)
plt.ylabel("Score")
plt.title("Final Test ROUGE Scores")
plt.grid(axis="y")
plt.tight_layout()
plt.savefig("./results/plots/rouge_comparison.png", dpi=200)
plt.show()

# 4. Precision, Recall and F1
plt.figure(figsize=(8, 5))
plt.bar(
    ["Precision", "Recall", "F1"],
    [evaluation_results["token_precision"],
     evaluation_results["token_recall"],
     evaluation_results["token_f1"]]
)
plt.ylabel("Score")
plt.title("Auxiliary Token-Overlap Metrics")
plt.ylim(0, 1)
plt.grid(axis="y")
plt.tight_layout()
plt.savefig("./results/plots/precision_recall_f1.png", dpi=200)
plt.show()

# 5. MSE and RMSE
plt.figure(figsize=(8, 5))
plt.bar(
    ["Token-ID MSE", "Token-ID RMSE"],
    [evaluation_results["token_id_mse"], evaluation_results["token_id_rmse"]]
)
plt.ylabel("Value")
plt.title("Auxiliary Token-ID Error Metrics")
plt.grid(axis="y")
plt.tight_layout()
plt.savefig("./results/plots/mse_rmse.png", dpi=200)
plt.show()

print("All plots saved in ./results/plots/")

In [ ]:
model_save_path = "./final_t5_summarization_model"
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print("Model and tokenizer saved successfully.")
print(os.listdir(model_save_path))

In [ ]:
# One unseen test example for qualitative comparison.
sample = dataset["test"][0]
article = sample["article"]
reference = sample["highlights"]

inputs = tokenizer(
    "summarize: " + article,
    return_tensors="pt",
    max_length=512,
    truncation=True
)
inputs = {key: value.to(device) for key, value in inputs.items()}

with torch.no_grad():
    summary_ids = model.generate(
        **inputs,
        max_length=128,
        min_length=30,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

generated_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("ORIGINAL ARTICLE:")
print(article)
print("\nREFERENCE SUMMARY:")
print(reference)
print("\nGENERATED SUMMARY:")
print(generated_summary)


# Long-text chunking used by the application.
import re

def split_into_chunks(text, max_tokens=450):
    sentences = re.split(r'(?<=[.!?])\s+', text.replace("\n", " ").strip())
    chunks = []
    current_sentences = []
    current_tokens = 0

    for sentence in sentences:
        if not sentence.strip():
            continue

        sentence_tokens = len(tokenizer.encode(sentence, add_special_tokens=False))

        if current_sentences and current_tokens + sentence_tokens > max_tokens:
            chunks.append(" ".join(current_sentences))
            current_sentences = []
            current_tokens = 0

        if sentence_tokens > max_tokens:
            token_ids = tokenizer.encode(sentence, add_special_tokens=False)
            for start in range(0, len(token_ids), max_tokens):
                chunks.append(
                    tokenizer.decode(token_ids[start:start + max_tokens], skip_special_tokens=True)
                )
            continue

        current_sentences.append(sentence)
        current_tokens += sentence_tokens

    if current_sentences:
        chunks.append(" ".join(current_sentences))

    return chunks

print("Long-text chunking function ready.")

In [ ]:
%%writefile app.py
import os
import re
import streamlit as st
import torch
from transformers import T5ForConditionalGeneration, AutoTokenizer

st.set_page_config(page_title="AI Text Summarizer", page_icon="📝", layout="wide")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "./final_t5_summarization_model"

@st.cache_resource
def load_model():
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"Model directory not found: {MODEL_PATH}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = T5ForConditionalGeneration.from_pretrained(MODEL_PATH)
    model = model.to(DEVICE)
    model.eval()
    return tokenizer, model

tokenizer, model = load_model()

def split_into_chunks(text, max_tokens=450):
    sentences = re.split(r'(?<=[.!?])\s+', text.replace("\n", " ").strip())
    chunks, current_sentences, current_tokens = [], [], 0

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        sentence_tokens = len(tokenizer.encode(sentence, add_special_tokens=False))

        if current_sentences and current_tokens + sentence_tokens > max_tokens:
            chunks.append(" ".join(current_sentences))
            current_sentences, current_tokens = [], 0

        if sentence_tokens > max_tokens:
            token_ids = tokenizer.encode(sentence, add_special_tokens=False)
            for start in range(0, len(token_ids), max_tokens):
                chunks.append(
                    tokenizer.decode(token_ids[start:start + max_tokens], skip_special_tokens=True)
                )
            continue

        current_sentences.append(sentence)
        current_tokens += sentence_tokens

    if current_sentences:
        chunks.append(" ".join(current_sentences))

    return chunks

def summarize_chunk(text):
    inputs = tokenizer("summarize: " + text, return_tensors="pt", max_length=512, truncation=True)
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

    with torch.no_grad():
        summary_ids = model.generate(
            **inputs,
            max_length=128,
            min_length=30,
            num_beams=4,
            length_penalty=1.0,
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

def summarize_long_text(text):
    chunks = split_into_chunks(text)

    if not chunks:
        return "", 0

    summaries = []
    progress = st.progress(0)
    status = st.empty()

    for i, chunk in enumerate(chunks):
        status.write(f"Summarizing part {i + 1} of {len(chunks)}...")
        summary = summarize_chunk(chunk)
        if summary.strip():
            summaries.append(summary)
        progress.progress((i + 1) / len(chunks))

    progress.empty()
    status.empty()

    if len(summaries) == 1:
        return summaries[0], len(chunks)

    combined = " ".join(summaries)
    combined_tokens = tokenizer.encode(combined, add_special_tokens=False)

    if len(combined_tokens) <= 450:
        return combined, len(chunks)

    return summarize_chunk(combined), len(chunks)

st.title("📝 AI Text Summarization Using Deep Learning")
st.write("Abstractive text summarization using a fine-tuned T5-small Transformer.")

article = st.text_area(
    "Enter your article or text:",
    height=400,
    placeholder="Paste your article here..."
)

if article.strip():
    col1, col2, col3 = st.columns(3)
    col1.metric("Characters", len(article))
    col2.metric("Words", len(article.split()))
    col3.metric("Approx. Tokens", len(tokenizer.encode(article, add_special_tokens=False)))

if st.button("Generate Summary", type="primary"):
    if not article.strip():
        st.warning("Please enter some text first.")
    else:
        try:
            with st.spinner("Generating summary..."):
                summary, chunk_count = summarize_long_text(article)

            st.subheader("Generated Summary")
            st.success(summary)

            original_words = len(article.split())
            summary_words = len(summary.split())
            ratio = summary_words / original_words if original_words else 0

            st.subheader("Summary Statistics")
            c1, c2, c3 = st.columns(3)
            c1.metric("Summary Words", summary_words)
            c2.metric("Chunks Processed", chunk_count)
            c3.metric("Summary / Original", f"{ratio:.2%}")

        except Exception as e:
            st.error(f"Unable to generate the summary: {e}")

with st.expander("Model Information"):
    st.write("Model: T5-small")
    st.write("Dataset: CNN/DailyMail")
    st.write("Task: Abstractive Text Summarization")
    st.write("Input limit per T5 call: 512 tokens")
    st.write("Output limit: 128 tokens")
    st.write(f"Device: {DEVICE}")

with st.expander("Evaluation Metrics"):
    results_file = "./results/evaluation_results.csv"
    if os.path.exists(results_file):
        import pandas as pd
        results = pd.read_csv(results_file)
        st.dataframe(results, use_container_width=True)
        st.caption(
            "Precision, Recall and F1 are token-overlap auxiliary metrics. "
            "Token-ID MSE/RMSE are numeric diagnostics and are not semantic "
            "quality measures for generated text."
        )
    else:
        st.info("Run the evaluation cells first to generate evaluation results.")

with st.expander("Training and Evaluation Plots"):
    plot_files = [
        ("Training / Validation Loss", "./results/plots/loss_curve.png"),
        ("ROUGE Progress", "./results/plots/rouge_curve.png"),
        ("Final ROUGE Scores", "./results/plots/rouge_comparison.png"),
        ("Precision / Recall / F1", "./results/plots/precision_recall_f1.png"),
        ("MSE / RMSE", "./results/plots/mse_rmse.png")
    ]

    for title, path in plot_files:
        if os.path.exists(path):
            st.write(f"### {title}")
            st.image(path)
        else:
            st.info(f"{title} is not available yet.")


In [ ]:
!pip install -q streamlit

In [ ]:
!pkill -f "streamlit run" || true

!nohup streamlit run app.py \
    --server.port 8501 \
    --server.address 0.0.0.0 \
    --server.headless true \
    > /content/streamlit.log 2>&1 &

In [ ]:
!nohup streamlit run app.py --server.port 8501 --server.headless true > /content/streamlit.log 2>&1 &
import time
time.sleep(5)
!curl -I http://localhost:8501

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
!./cloudflared tunnel --url http://localhost:8501
